[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/REPO/blob/main/notebooks/lec03-keras.ipynb)

# Aula 3 — Transfer Learning com TensorFlow/Keras (CIFAR-10)

Notebook de acompanhamento da Aula 3 — **Tópicos Avançados em IA · UFABC**  
Adaptado de MIT 15.773 (Farias & Ramakrishnan, 2024)

---

## Índice

1. [O que é Transfer Learning?](#1-o-que-é-transfer-learning)
2. [Configuração](#2-configuração)
3. [Dataset CIFAR-10](#3-dataset-cifar-10)
4. [Pré-processamento com `tf.data`](#4-pré-processamento-com-tfdata)
5. [Fase 1 — Extração de Características](#5-fase-1--extração-de-características)
6. [Fase 2 — Fine-tuning](#6-fase-2--fine-tuning)
7. [Resultados](#7-resultados)
8. [Predições](#8-predições)

## 1. O que é Transfer Learning?

**Transfer Learning** (aprendizado por transferência) reutiliza o conhecimento adquirido em uma tarefa de grande escala — normalmente classificação do ImageNet (1,2 M imagens, 1 000 classes) — para resolver uma tarefa diferente, com muito menos dados e tempo de treinamento.

### Intuição

Redes convolucionais profundas aprendem uma hierarquia de representações:

| Camadas iniciais | Camadas intermediárias | Camadas finais |
|---|---|---|
| Bordas, gradientes | Texturas, padrões | Partes de objetos, semântica |

As representações iniciais são **universais** (aplicáveis a qualquer imagem); as finais são **específicas** ao domínio de treinamento original.  
Por isso podemos congelar as primeiras camadas e adaptar apenas as últimas.

### Duas estratégias

| Estratégia | Quando usar | O que é treinável |
|---|---|---|
| **Extração de características** (*feature extraction*) | Poucos dados, tarefa similar | Apenas a cabeça (*head*) adicionada |
| **Fine-tuning** | Dados moderados, mais controle | Cabeça + últimas camadas da base |

> Neste notebook seguiremos as **duas fases em sequência**, como recomendado pelo guia oficial do Keras.

### Modelo base: MobileNetV2

Arquitetura leve otimizada para dispositivos móveis:  
$$\text{MobileNetV2}: \mathscr{F}(x) = \text{depthwise-separable convolutions} + \text{inverted residuals}$$

Pré-treinado no ImageNet, entrada mínima de 96×96 pixels.

## 2. Configuração

In [ ]:
# @title Instala dependências (necessário no Colab)
# !pip install -q tensorflow

In [ ]:
# @title Imports
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

import warnings
warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU disponível: {bool(tf.config.list_physical_devices("GPU"))}')

## 3. Dataset CIFAR-10

60 000 imagens coloridas 32×32 divididas em 10 classes.  
Split padrão: **50 000 treino / 10 000 teste**.

| Índice | Classe | Índice | Classe |
|---|---|---|---|
| 0 | avião | 5 | cão |
| 1 | automóvel | 6 | sapo |
| 2 | pássaro | 7 | cavalo |
| 3 | gato | 8 | navio |
| 4 | cervo | 9 | caminhão |

In [ ]:
# @title Carrega CIFAR-10
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

CLASS_NAMES = ['avião','automóvel','pássaro','gato','cervo',
               'cão','sapo','cavalo','navio','caminhão']

print(f'Treino : {x_train.shape}  dtype={x_train.dtype}')
print(f'Teste  : {x_test.shape}')

In [ ]:
# @title Visualiza amostras
fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(x_train[i])
    ax.set_title(CLASS_NAMES[int(y_train[i])], fontsize=9)
    ax.axis('off')
plt.suptitle('Amostras do CIFAR-10 (imagens 32×32)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Pré-processamento com `tf.data`

O MobileNetV2 espera:
- Imagens **96×96** pixels (mínimo aceito pelo modelo)
- Valores no intervalo **[-1, 1]** — aplicado pela função `mobilenet_v2.preprocess_input`

Usamos o pipeline `tf.data.Dataset` para:
1. **`.map()`** — redimensiona e normaliza em paralelo
2. **`.shuffle()`** — embaralha antes de cada época
3. **`.batch()`** — agrupa em mini-lotes de 64
4. **`.prefetch(AUTOTUNE)`** — carrega o próximo lote enquanto a GPU processa o atual

> O pipeline `tf.data` é executado em CPU de forma assíncrona, mantendo a GPU ocupada ao máximo.

In [ ]:
# @title Constantes do pipeline
IMG_SIZE   = 96
BATCH_SIZE = 64
AUTOTUNE   = tf.data.AUTOTUNE

In [ ]:
# @title Funções de pré-processamento
def preprocess(image, label):
    """Redimensiona para IMG_SIZE x IMG_SIZE e normaliza para [-1, 1]."""
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label

def augment(image, label):
    """Aumentação leve aplicada somente ao conjunto de treino."""
    image, label = preprocess(image, label)
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    return image, label

In [ ]:
# @title Constrói os datasets
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .shuffle(buffer_size=10_000)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print('Dataset de treino :', train_ds)
print('Dataset de teste  :', test_ds)

## 5. Fase 1 — Extração de Características

### Estratégia

Congelamos **todos** os pesos da MobileNetV2 (`base_model.trainable = False`) e treinamos apenas a cabeça de classificação adicionada ao topo.

**Por quê começar assim?**  
Se treinarmos todas as camadas de uma vez com pesos aleatórios na cabeça, os gradientes grandes da cabeça podem destruir as representações pré-treinadas (fenômeno chamado de *catastrophic forgetting*).

### Arquitetura adicionada

```
MobileNetV2 (congelada)  →  GlobalAveragePooling2D  →  Dropout(0.2)  →  Dense(10, softmax)
```

O `GlobalAveragePooling2D` comprime o mapa de características espacial `(3, 3, 1280)` em um vetor `(1280,)`, funcionando como um descritor compacto da imagem:
$$\mathscr{GAP}(F)_k = \frac{1}{H \times W} \sum_{i,j} F_{i,j,k}$$

In [ ]:
# @title Carrega MobileNetV2 sem a cabeça classificadora
base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False

print(f'Camadas totais na base   : {len(base_model.layers)}')
print(f'Parâmetros treináveis    : {base_model.count_params():,}')

In [ ]:
# @title Constrói o modelo completo (Fase 1)
inputs  = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = base_model(inputs, training=False)   # training=False mantém BatchNorm congelado
x       = keras.layers.GlobalAveragePooling2D()(x)
x       = keras.layers.Dropout(0.2)(x)
outputs = keras.layers.Dense(10, activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.summary(show_trainable=True)

In [ ]:
# @title Compila — Fase 1
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

In [ ]:
# @title Treina — Fase 1 (5 épocas)
EPOCHS_1 = 5

history1 = model.fit(
    train_ds,
    epochs=EPOCHS_1,
    validation_data=test_ds,
    verbose=1,
)

print(f'\nAcurácia de validação (Fase 1): {history1.history["val_accuracy"][-1]:.1%}')

## 6. Fase 2 — Fine-tuning

Com a cabeça já treinada, podemos **descongelar as últimas camadas** da base e ajustar todo o modelo com uma taxa de aprendizado muito menor.

### Por que apenas as últimas camadas?

- Camadas iniciais aprendem características de baixo nível (universais) → **não precisam mudar**
- Camadas finais codificam semântica de alto nível específica do ImageNet → **vale a pena adaptar ao CIFAR-10**

### Taxa de aprendizado menor (1e-4)

Com o modelo já convergido na Fase 1, uma taxa grande causaria instabilidade.  
Usamos $\eta = 10^{-4}$, dez vezes menor que na Fase 1:

$$\mathscr{L}_{\text{fine-tune}} \approx \mathscr{L}_{\text{fase 1}} - \Delta, \quad \Delta \ll \mathscr{L}_{\text{fase 1}}$$

In [ ]:
# @title Descongela as últimas 30 camadas da base
base_model.trainable = True

# Congela tudo exceto as últimas 30 camadas
freeze_until = len(base_model.layers) - 30
for layer in base_model.layers[:freeze_until]:
    layer.trainable = False

trainable = sum(1 for l in base_model.layers if l.trainable)
frozen    = sum(1 for l in base_model.layers if not l.trainable)
print(f'Camadas treináveis na base: {trainable}')
print(f'Camadas congeladas na base: {frozen}')

In [ ]:
# @title Re-compila com taxa de aprendizado menor — Fase 2
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary(show_trainable=True)

In [ ]:
# @title Treina — Fase 2 (5 épocas adicionais)
EPOCHS_2 = 5

history2 = model.fit(
    train_ds,
    epochs=EPOCHS_1 + EPOCHS_2,
    initial_epoch=EPOCHS_1,        # continua de onde parou
    validation_data=test_ds,
    verbose=1,
)

print(f'\nAcurácia de validação (Fase 2): {history2.history["val_accuracy"][-1]:.1%}')

## 7. Resultados

Comparamos as duas fases em um único gráfico de acurácia.  
A linha tracejada vertical marca o início do fine-tuning.

In [ ]:
# @title Concatena históricos e plota
acc_all     = history1.history['accuracy']     + history2.history['accuracy']
val_acc_all = history1.history['val_accuracy'] + history2.history['val_accuracy']

plt.figure(figsize=(10, 5))
plt.plot(acc_all,     label='Treino',     linewidth=2)
plt.plot(val_acc_all, label='Validação',  linewidth=2)
plt.axvline(x=EPOCHS_1 - 0.5, color='gray', linestyle='--', linewidth=1.5,
            label='Início do fine-tuning')
plt.fill_betweenx([0, 1], 0, EPOCHS_1 - 0.5,
                  alpha=0.05, color='blue',  label='Extração de características')
plt.fill_betweenx([0, 1], EPOCHS_1 - 0.5, len(acc_all),
                  alpha=0.05, color='orange', label='Fine-tuning')
plt.title('Acurácia — Transfer Learning no CIFAR-10', fontsize=13)
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.legend(loc='lower right')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# @title Avaliação final no conjunto de teste
loss_final, acc_final = model.evaluate(test_ds, verbose=0)
print(f'Loss final no teste   : {loss_final:.4f}')
print(f'Acurácia final no teste: {acc_final:.1%}')

## 8. Predições

Visualizamos 9 imagens do conjunto de teste com o rótulo predito e o real.  
**Verde** = predição correta · **Vermelho** = predição errada.

In [ ]:
# @title Grade 3×3 de predições
# Obtém um lote do conjunto de teste
sample_images, sample_labels = next(iter(
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(9)
))

preds = model.predict(sample_images, verbose=0)
pred_labels = np.argmax(preds, axis=1)
true_labels = sample_labels.numpy().ravel()

# Imagens originais (sem normalização) para exibição
raw_images = x_test[:9]

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(raw_images[i])
    correct = (pred_labels[i] == true_labels[i])
    color   = 'green' if correct else 'red'
    conf    = preds[i][pred_labels[i]]
    ax.set_title(
        f'Pred: {CLASS_NAMES[pred_labels[i]]} ({conf:.0%})\n'
        f'Real: {CLASS_NAMES[true_labels[i]]}',
        color=color, fontsize=10
    )
    ax.axis('off')

plt.suptitle('Predições no conjunto de teste\n(verde = correto, vermelho = errado)',
             fontsize=13)
plt.tight_layout()
plt.show()

---

## Resumo

| Fase | Camadas treináveis | LR | Épocas |
|---|---|---|---|
| Extração de características | Somente cabeça (~10 K params) | 1e-3 | 5 |
| Fine-tuning | Cabeça + últimas 30 camadas da base | 1e-4 | 5 |

### Pontos-chave

1. **Congele antes de ajustar** — treinar a cabeça primeiro estabiliza os gradientes
2. **Taxa de aprendizado menor no fine-tuning** — preserva o conhecimento pré-treinado
3. **`training=False` no base_model** — mantém as camadas de BatchNorm em modo de inferência mesmo quando os pesos estão congelados
4. **`tf.data` + `prefetch`** — pipeline assíncrono que evita gargalos de I/O

### Próximos passos sugeridos

- Experimentar outros modelos base (EfficientNetV2, ResNet50, DenseNet121)
- Adicionar *learning rate scheduling* (cosine decay, warm-up)
- Testar em um dataset customizado com apenas algumas centenas de imagens por classe
- Usar `keras.callbacks.ModelCheckpoint` para salvar o melhor modelo